# Part 2 · Meaning as geometry

**Building Agentic AI — Day 1, 10:10–11:00**

In Part 1 the model turned text into text. In this part we turn text into
**numbers** — vectors — and discover that *similar meanings land close together*.
This single idea powers semantic search, RAG, and your Day-3 agent's retrieval tool.

Plan: a 10-minute `numpy` speedrun → dot products & cosine similarity by hand →
real Gemini embeddings.

In [ ]:
import os
import time

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env", override=True)
client = genai.Client()

EMBED_MODEL = "gemini-embedding-001"

## 1. numpy speedrun (only what we need)

`numpy` arrays are the substrate of all machine learning: fixed-type grids of
numbers with *fast* math.

In [ ]:
v = np.array([1.0, 3.0, -2.0])

print("vector:", v)
print("shape:", v.shape, "· dtype:", v.dtype)
print("v * 2:", v * 2)          # elementwise — no loop needed
print("v + v:", v + v)

The reason everyone uses numpy — vectorized math runs in optimized C, not in the
Python interpreter:

In [ ]:
big = np.random.default_rng(0).random(2_000_000)

t0 = time.perf_counter()
total_loop = sum(x * x for x in big)          # plain Python
t1 = time.perf_counter()
total_np = np.dot(big, big)                   # numpy
t2 = time.perf_counter()

print(f"python loop: {t1 - t0:.3f}s · numpy: {t2 - t1:.4f}s · same answer: {np.isclose(total_loop, total_np)}")

One more tool we'll need this afternoon: **finding the best entries** in an array.

In [ ]:
scores = np.array([0.12, 0.94, 0.51, 0.88, 0.07])

print("highest value:", scores.max())
print("index of highest:", scores.argmax())
print("indices, best→worst:", np.argsort(scores)[::-1])   # argsort is ascending, so we flip it

`argsort` + slicing = "top-k results" — that *is* the heart of a search engine.
Remember it for Part 3.

## 2. Vectors as coordinates of meaning — a toy universe

Forget AI for a second. Let's rate a few games *by hand* on just two axes:
how **cozy** and how **action-packed** they are (0 to 1).

In [ ]:
import matplotlib.pyplot as plt

toy = {
    "Dungeon Sprouts":   [0.95, 0.30],   # [coziness, action]
    "Rooftop Ramen":     [0.90, 0.35],
    "Neon Drift Racers": [0.15, 0.90],
    "Speedrun Sandwich": [0.40, 0.85],
    "Silent Depths":     [0.05, 0.60],
    "Bureau of Time":    [0.55, 0.25],
}

fig, ax = plt.subplots(figsize=(7, 5))
for name, (x, y) in toy.items():
    ax.scatter(x, y, s=80)
    ax.annotate(name, (x, y), xytext=(6, 4), textcoords="offset points")
ax.set_xlabel("coziness →")
ax.set_ylabel("action →")
ax.set_title("Six games in 'meaning space' (hand-made, 2 dimensions)")
plt.show()

Games with a similar *vibe* sit close together. That closeness is something we
can **compute** — and that's the entire trick.

Real embedding models do exactly this, except:

- the axes are **learned from data**, not chosen by us,
- there are **hundreds to thousands** of them, not 2,
- and no single axis means anything readable — only *distances* matter.

## 3. Dot product → cosine similarity, by hand

The **dot product** multiplies vectors element-by-element and sums:
$a \cdot b = \sum_i a_i b_i$. It's big when vectors point the same way.

In [ ]:
a = np.array(toy["Dungeon Sprouts"])
b = np.array(toy["Rooftop Ramen"])
c = np.array(toy["Neon Drift Racers"])

manual = a[0] * b[0] + a[1] * b[1]
print("by hand:  ", manual)
print("np.dot:   ", np.dot(a, b))
print("vs racer: ", np.dot(a, c))

One flaw: the dot product also grows with vector *length*, so "loud" vectors win
even when their direction is off. Fix: compare **directions only** — divide by both
lengths. That's **cosine similarity**:

$\text{cos}(a,b) = \dfrac{a \cdot b}{\lVert a\rVert \, \lVert b \rVert} \in [-1, 1]$

`+1` same direction · `0` unrelated · `−1` opposite.

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


print("cozy vs cozy:     ", round(cosine(a, b), 3))
print("cozy vs racer:    ", round(cosine(a, c), 3))
print("v vs 100*v (same direction):", round(cosine(a, a * 100), 3))

You just wrote the function that ranks every search result, RAG lookup and
recommendation this week. Four lines, no framework.

## 4. Real embeddings from Gemini

Now we let a trained model place **whole sentences** into a 768-dimensional
meaning space.

In [ ]:
sentences = [
    "The boss fight was brutally difficult but fair.",
    "I couldn't beat the final boss, way too hard.",
    "The soundtrack is absolutely gorgeous.",
    "Muzica din jocul ăsta e superbă.",
    "The game crashes every time I open the map.",
    "Constant freezes whenever I fast-travel.",
    "Best co-op experience I've had in years.",
    "My sourdough bread came out perfect today.",
]

result = client.models.embed_content(
    model=EMBED_MODEL,
    contents=sentences,
    config=types.EmbedContentConfig(
        task_type="SEMANTIC_SIMILARITY",
        output_dimensionality=768,
    ),
)

E = np.array([e.values for e in result.embeddings])
E = E / np.linalg.norm(E, axis=1, keepdims=True)   # normalize → cosine becomes a plain dot product
print("embeddings shape:", E.shape)

Each sentence is now a point in 768-dimensional space. Here's what "meaning"
looks like as numbers (first 8 of 768 coordinates):

In [ ]:
print(sentences[0])
print(np.round(E[0, :8], 3), "...")

Since every row is normalized, `E @ E.T` computes the cosine similarity of
**every pair at once** — your `cosine()` function, vectorized:

In [ ]:
import pandas as pd

labels = ["boss-hard", "boss-cant", "soundtrack", "muzica-RO", "crash-map", "freeze-travel", "coop", "sourdough"]
sim = pd.DataFrame(E @ E.T, index=labels, columns=labels)
sim.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).format("{:.2f}")

Read the matrix — three things should jump out:

1. **Synonyms cluster**: *crash-map* ↔ *freeze-travel* score high with zero shared words.
2. **Languages don't matter**: *soundtrack* ↔ *muzica-RO* — the model maps Romanian
   and English into the **same** meaning space.
3. **Sourdough is lonely**: lowest similarity against everything. As it should be.

And a preview of this afternoon — finding the closest sentence to a query is
`argsort` on one matrix-vector product:

In [ ]:
query = "the game keeps dying on startup"

q = np.array(
    client.models.embed_content(
        model=EMBED_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY", output_dimensionality=768),
    ).embeddings[0].values
)
q = q / np.linalg.norm(q)

for i in np.argsort(E @ q)[::-1][:3]:
    print(f"{(E @ q)[i]:.3f}  {sentences[i]}")

## 5. Exercises

**5.1 — Odd one out (⭐)** Write `odd_one_out(sentences)` that embeds a list and
returns the sentence with the *lowest average similarity* to the others.
Test it on `sentences` above (we all know who it is), then invent a sneakier list.

**5.2 — Vectorized cosine matrix (⭐⭐)** Write `cosine_matrix(A)` that computes
all-pairs cosine similarity for *any* (unnormalized) matrix `A`, without loops.
Verify against nested `cosine()` calls using `np.allclose`.

**5.3 — Your languages (⭐)** Write the same sentence in two languages you speak,
plus one unrelated sentence. Check that meaning beats language.

**5.4 — Shrinking dimensions (⭐⭐)** Re-embed `sentences` with
`output_dimensionality=128`. Does the *ranking* of similarities change much?
(Gemini embeddings are trained so prefixes of the vector still work — smaller,
cheaper, slightly blurrier.)

In [ ]:
# 5.1 — your code here

In [ ]:
# 5.2 — your code here

In [ ]:
# 5.3 — your code here

In [ ]:
# 5.4 — your code here

---
## ✅ Checkpoint

You can now: use numpy arrays, compute cosine similarity *from scratch*, and turn
any text into a vector where distance = meaning.

After lunch we scale this up: semantic search over
**300 player reviews** — in about 15 lines of code.